# NIH Dataset

In [ ]:
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/CVP/kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
import kagglehub
import json
import shutil
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from pathlib import Path
import cv2
import numpy as np
import pydicom
from pydicom.pixel_data_handlers.util import apply_modality_lut, apply_voi_lut



# Download latest version
path = kagglehub.dataset_download("nih-chest-xrays/data")

print("Path to dataset files:", path)

In [ ]:
FULL_NIH_ROOT = Path("/root/.cache/kagglehub/datasets/nih-chest-xrays/data/versions/3")

DATA_ENTRY = FULL_NIH_ROOT / "Data_Entry_2017.csv"
BBOX_CSV = FULL_NIH_ROOT / "BBox_List_2017.csv"

assert FULL_NIH_ROOT.exists(), f"Missing NIH root: {FULL_NIH_ROOT}"
assert DATA_ENTRY.exists(), f"Missing: {DATA_ENTRY}"
assert BBOX_CSV.exists(), f"Missing: {BBOX_CSV}"

OUT_ROOT = Path("/content/datasets/NIH_Extracted")
OUT_IMG_DIR = OUT_ROOT / "images"
OUT_CSV_DIR = OUT_ROOT / "csv"

OUT_IMG_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV_DIR.mkdir(parents=True, exist_ok=True)

data = pd.read_csv(DATA_ENTRY)
bbox = pd.read_csv(BBOX_CSV)

# Map image filenames to actual full NIH image paths
all_image_paths = list(FULL_NIH_ROOT.glob("images_*/images/*.png"))
path_map = {p.name: str(p) for p in all_image_paths}

print("Total full NIH image files found:", len(path_map))
print("Data_Entry rows:", len(data))
print("BBox rows:", len(bbox))

data = data[data["Image Index"].isin(path_map)].copy()
data["source_image"] = data["Image Index"].map(path_map)

# Clean bbox columns
bbox = bbox.rename(columns={
    "Image Index": "image_name",
    "Finding Label": "finding_label",
    "Bbox [x": "x",
    "y": "y",
    "w": "width",
    "h]": "height",
})

bbox = bbox[["image_name", "finding_label", "x", "y", "width", "height"]].copy()

# Positive test: Pneumonia bbox images
pneu_bbox = bbox[bbox["finding_label"] == "Pneumonia"].copy()

pneu_bbox_grouped = pneu_bbox.groupby("image_name", as_index=False).agg({
    "finding_label": "first",
    "x": list,
    "y": list,
    "width": list,
    "height": list,
})

pneu_bbox_grouped["source_image"] = pneu_bbox_grouped["image_name"].map(path_map)
pneu_bbox_grouped = pneu_bbox_grouped[pneu_bbox_grouped["source_image"].notna()].copy()

print("Pneumonia bbox test images:", len(pneu_bbox_grouped))

def make_boxes(row):
    boxes = []
    for x, y, w, h in zip(row["x"], row["y"], row["width"], row["height"]):
        boxes.append({
            "x": float(x),
            "y": float(y),
            "width": float(w),
            "height": float(h),
        })
    return boxes

pneu_test = pd.DataFrame({
    "source_image": pneu_bbox_grouped["source_image"],
    "label": 1,
    "class": "Pneumonia",
    "Image Index": pneu_bbox_grouped["image_name"],
    "Finding Labels": "Pneumonia",
    "has_bbox": True,
    "boxes_json": pneu_bbox_grouped.apply(make_boxes, axis=1).apply(json.dumps),
})

# Negative test: same number of No Finding images
N_TEST_POS = len(pneu_test)

nofinding_all = data[data["Finding Labels"] == "No Finding"].copy()

# avoid accidentally selecting bbox images
bbox_image_names = set(bbox["image_name"].unique())
nofinding_candidates = nofinding_all[
    ~nofinding_all["Image Index"].isin(bbox_image_names)
].copy()

neg_test = nofinding_candidates.sample(n=N_TEST_POS, random_state=42).copy()

neg_test = pd.DataFrame({
    "source_image": neg_test["source_image"],
    "label": 0,
    "class": "Normal",
    "Image Index": neg_test["Image Index"],
    "Finding Labels": neg_test["Finding Labels"],
    "has_bbox": False,
    "boxes_json": json.dumps([]),
})

test_df = pd.concat([pneu_test, neg_test], axis=0)
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)

test_image_names = set(test_df["Image Index"])

print("\nFinal test set:")
print(test_df["label"].value_counts())
print(test_df["class"].value_counts())
print("Total test:", len(test_df))

remaining = data[~data["Image Index"].isin(test_image_names)].copy()

pos_trainval = remaining[
    remaining["Finding Labels"].str.split("|").apply(lambda labels: "Pneumonia" in labels)
].copy()

neg_trainval = remaining[
    remaining["Finding Labels"] == "No Finding"
].copy()

print("\nTrain/valid candidates before balancing:")
print("Pneumonia:", len(pos_trainval))
print("No Finding:", len(neg_trainval))

# Balance negatives to positives
N_POS = len(pos_trainval)
neg_trainval_sample = neg_trainval.sample(n=N_POS, random_state=42).copy()

pos_trainval["label"] = 1
pos_trainval["class"] = "Pneumonia"

neg_trainval_sample["label"] = 0
neg_trainval_sample["class"] = "Normal"

trainval_df = pd.concat([pos_trainval, neg_trainval_sample], axis=0)
trainval_df = trainval_df.sample(frac=1, random_state=42).reset_index(drop=True)

trainval_df["has_bbox"] = False
trainval_df["boxes_json"] = json.dumps([])

trainval_df = trainval_df[
    ["source_image", "label", "class", "Image Index", "Finding Labels", "has_bbox", "boxes_json"]
].copy()

train_df, valid_df = train_test_split(
    trainval_df,
    test_size=0.2,
    random_state=42,
    stratify=trainval_df["label"]
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

print("\nSplit counts:")
print("Train:", len(train_df))
print(train_df["label"].value_counts())
print("Valid:", len(valid_df))
print(valid_df["label"].value_counts())
print("Test:", len(test_df))
print(test_df["label"].value_counts())

all_needed = pd.concat([
    train_df[["source_image", "Image Index"]],
    valid_df[["source_image", "Image Index"]],
    test_df[["source_image", "Image Index"]],
], axis=0).drop_duplicates("Image Index").reset_index(drop=True)

print("\nUnique images to copy:", len(all_needed))

for _, row in tqdm(all_needed.iterrows(), total=len(all_needed), desc="Copying images"):
    src = Path(row["source_image"])
    dst = OUT_IMG_DIR / row["Image Index"]

    if not dst.exists():
        shutil.copy2(src, dst)

print("Copied images to:", OUT_IMG_DIR)

def finalize_csv(df):
    df = df.copy()
    df["image"] = df["Image Index"].apply(lambda x: str(OUT_IMG_DIR / x))
    df = df[[
        "image",
        "label",
        "class",
        "Image Index",
        "Finding Labels",
        "has_bbox",
        "boxes_json",
    ]]
    return df

train_final = finalize_csv(train_df)
valid_final = finalize_csv(valid_df)
test_final = finalize_csv(test_df)

train_final.to_csv(OUT_CSV_DIR / "nih_pneumonia_train.csv", index=False)
valid_final.to_csv(OUT_CSV_DIR / "nih_pneumonia_valid.csv", index=False)
test_final.to_csv(OUT_CSV_DIR / "nih_pneumonia_test.csv", index=False)

# Also copy raw metadata files for safety/reproducibility
shutil.copy2(DATA_ENTRY, OUT_CSV_DIR / "Data_Entry_2017.csv")
shutil.copy2(BBOX_CSV, OUT_CSV_DIR / "BBox_List_2017.csv")

manifest = {
    "source_root": str(FULL_NIH_ROOT),
    "output_root": str(OUT_ROOT),
    "num_images_copied": int(len(all_needed)),
    "train_csv": str(OUT_CSV_DIR / "nih_pneumonia_train.csv"),
    "valid_csv": str(OUT_CSV_DIR / "nih_pneumonia_valid.csv"),
    "test_csv": str(OUT_CSV_DIR / "nih_pneumonia_test.csv"),
    "train_size": int(len(train_final)),
    "valid_size": int(len(valid_final)),
    "test_size": int(len(test_final)),
    "test_positive_pneumonia_with_bbox": int((test_final["label"] == 1).sum()),
    "test_negative_normal_no_bbox": int((test_final["label"] == 0).sum()),
}

with open(OUT_ROOT / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("\nDone.")
print("Extracted dataset saved to:", OUT_ROOT)
print("\nFiles:")
print(OUT_CSV_DIR / "nih_pneumonia_train.csv")
print(OUT_CSV_DIR / "nih_pneumonia_valid.csv")
print(OUT_CSV_DIR / "nih_pneumonia_test.csv")
print(OUT_ROOT / "manifest.json")

print("\nFinal CSV counts:")
print("Train:")
print(train_final["label"].value_counts())
print("Valid:")
print(valid_final["label"].value_counts())
print("Test:")
print(test_final["label"].value_counts())
print("\nTest bbox counts:")
print(test_final["has_bbox"].value_counts())

# RSNA Dataset

In [ ]:
path = kagglehub.competition_download('rsna-pneumonia-detection-challenge')

print("Path to competition files:", path)

In [ ]:
data_path = Path("/root/.cache/kagglehub/competitions/rsna-pneumonia-detection-challenge")

In [ ]:
dcm_dir = data_path / "stage_2_train_images"
png_dir = Path("/content/data/images_png")
png_dir.mkdir(parents=True, exist_ok=True)

def resize_and_save(load_path, save_path):
    ds = pydicom.dcmread(str(load_path), force=True)

    img = ds.pixel_array
    img = apply_modality_lut(img, ds)
    img = apply_voi_lut(img, ds)

    if hasattr(ds, "PhotometricInterpretation"):
        if ds.PhotometricInterpretation.lower().strip() == "monochrome1":
            img = img.max() - img

    h, w = img.shape
    ratio = 512 / min(h, w)
    target_size = (int(w * ratio), int(h * ratio))

    img = cv2.resize(img, target_size, cv2.INTER_LANCZOS4)

    img = img.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8) * 255
    img = img.astype(np.uint8)

    cv2.imwrite(str(save_path), img)

dcm_files = sorted(dcm_dir.glob("*.dcm"))

print("DICOM files:", len(dcm_files))
print("Saving PNGs to:", png_dir)

for p in tqdm(dcm_files):
    out = png_dir / f"{p.stem}.png"
    if not out.exists():
        resize_and_save(p, out)

print("PNG files:", len(list(png_dir.glob("*.png"))))

In [ ]:
RSNA_ROOT = Path("/root/.cache/kagglehub/competitions/rsna-pneumonia-detection-challenge")
PNG_SOURCE_DIR = Path("/content/data/images_png")

LABELS_CSV = RSNA_ROOT / "stage_2_train_labels.csv"
CLASS_CSV = RSNA_ROOT / "stage_2_detailed_class_info.csv"

assert RSNA_ROOT.exists(), f"Missing RSNA root: {RSNA_ROOT}"
assert PNG_SOURCE_DIR.exists(), f"Missing PNG folder: {PNG_SOURCE_DIR}"
assert LABELS_CSV.exists(), f"Missing: {LABELS_CSV}"
assert CLASS_CSV.exists(), f"Missing: {CLASS_CSV}"

OUT_ROOT = Path("/content/datasets/RSNA_Extracted")
OUT_IMG_DIR = OUT_ROOT / "images"
OUT_CSV_DIR = OUT_ROOT / "csv"

OUT_IMG_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV_DIR.mkdir(parents=True, exist_ok=True)

labels = pd.read_csv(LABELS_CSV)
class_info = pd.read_csv(CLASS_CSV)

print("Raw labels rows:", len(labels))
print("Class info rows:", len(class_info))
print("PNG files found:", len(list(PNG_SOURCE_DIR.glob("*.png"))))

label_df = labels.groupby("patientId")["Target"].max().reset_index()
label_df = label_df.rename(columns={"patientId": "id", "Target": "label"})

class_df = class_info[["patientId", "class"]].drop_duplicates("patientId")
class_df = class_df.rename(columns={"patientId": "id"})

df = label_df.merge(class_df, on="id", how="left")

# Repo-compatible class names
df["class"] = df["class"].apply(
    lambda x: "Pneumonia" if x == "Lung Opacity" else "Normal"
)

# Source PNG path
df["source_image"] = df["id"].apply(lambda x: str(PNG_SOURCE_DIR / f"{x}.png"))

# Keep only images that exist
df = df[df["source_image"].apply(lambda p: Path(p).exists())].reset_index(drop=True)

print("\nImage-level dataset:")
print(df[["id", "label", "class", "source_image"]].head())
print("\nLabel counts:")
print(df["label"].value_counts())
print("\nClass counts:")
print(df["class"].value_counts())
print("Total:", len(df))

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

def balance_split(split_df, random_state=42):
    min_count = split_df["label"].value_counts().min()
    balanced = (
        split_df.groupby("label", group_keys=False)
        .apply(lambda x: x.sample(n=min_count, random_state=random_state))
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )
    return balanced

train_balanced = balance_split(train_df)
valid_balanced = balance_split(valid_df)
test_balanced = balance_split(test_df)

print("\nBalanced split counts:")
print("Train:", len(train_balanced))
print(train_balanced["label"].value_counts())
print("Valid:", len(valid_balanced))
print(valid_balanced["label"].value_counts())
print("Test:", len(test_balanced))
print(test_balanced["label"].value_counts())

all_needed = pd.concat([
    train_balanced[["id", "source_image"]],
    valid_balanced[["id", "source_image"]],
    test_balanced[["id", "source_image"]],
], axis=0).drop_duplicates("id").reset_index(drop=True)

print("\nUnique images to copy:", len(all_needed))

for _, row in tqdm(all_needed.iterrows(), total=len(all_needed), desc="Copying RSNA images"):
    src = Path(row["source_image"])
    dst = OUT_IMG_DIR / f"{row['id']}.png"

    if not dst.exists():
        shutil.copy2(src, dst)

print("Copied images to:", OUT_IMG_DIR)

def finalize_csv(split_df):
    split_df = split_df.copy()
    split_df["image"] = split_df["id"].apply(lambda x: str(OUT_IMG_DIR / f"{x}.png"))
    split_df = split_df[["image", "label", "class", "id"]]
    return split_df

train_final = finalize_csv(train_balanced)
valid_final = finalize_csv(valid_balanced)
test_final = finalize_csv(test_balanced)

train_final.to_csv(OUT_CSV_DIR / "rsna_train.csv", index=False)
valid_final.to_csv(OUT_CSV_DIR / "rsna_valid.csv", index=False)
test_final.to_csv(OUT_CSV_DIR / "rsna_test.csv", index=False)

# Also save unbalanced versions, just in case
finalize_csv(train_df).to_csv(OUT_CSV_DIR / "rsna_train_unbalanced.csv", index=False)
finalize_csv(valid_df).to_csv(OUT_CSV_DIR / "rsna_valid_unbalanced.csv", index=False)
finalize_csv(test_df).to_csv(OUT_CSV_DIR / "rsna_test_unbalanced.csv", index=False)

# Copy raw annotation CSVs for XAI boxes/reproducibility
shutil.copy2(LABELS_CSV, OUT_CSV_DIR / "stage_2_train_labels.csv")
shutil.copy2(CLASS_CSV, OUT_CSV_DIR / "stage_2_detailed_class_info.csv")

manifest = {
    "source_rsna_root": str(RSNA_ROOT),
    "source_png_dir": str(PNG_SOURCE_DIR),
    "output_root": str(OUT_ROOT),
    "num_images_copied": int(len(all_needed)),
    "train_csv": str(OUT_CSV_DIR / "rsna_train.csv"),
    "valid_csv": str(OUT_CSV_DIR / "rsna_valid.csv"),
    "test_csv": str(OUT_CSV_DIR / "rsna_test.csv"),
    "train_size": int(len(train_final)),
    "valid_size": int(len(valid_final)),
    "test_size": int(len(test_final)),
    "train_counts": train_final["label"].value_counts().to_dict(),
    "valid_counts": valid_final["label"].value_counts().to_dict(),
    "test_counts": test_final["label"].value_counts().to_dict(),
    "raw_labels_csv": str(OUT_CSV_DIR / "stage_2_train_labels.csv"),
    "raw_class_csv": str(OUT_CSV_DIR / "stage_2_detailed_class_info.csv"),
}

with open(OUT_ROOT / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("\nDone.")
print("Extracted RSNA dataset saved to:", OUT_ROOT)
print("\nFiles:")
print(OUT_CSV_DIR / "rsna_train.csv")
print(OUT_CSV_DIR / "rsna_valid.csv")
print(OUT_CSV_DIR / "rsna_test.csv")
print(OUT_CSV_DIR / "stage_2_train_labels.csv")
print(OUT_CSV_DIR / "stage_2_detailed_class_info.csv")
print(OUT_ROOT / "manifest.json")

print("\nFinal counts:")
print("Train:")
print(train_final["label"].value_counts())
print("Valid:")
print(valid_final["label"].value_counts())
print("Test:")
print(test_final["label"].value_counts())